# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.

%load_ext dotenv
%dotenv 


In [2]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [6]:
import os
from glob import glob

# Write your code below.
# See the code below



In [4]:
# Loading the environment variable "PRICE_DATA"
PRICE_DATA =os.getenv('PRICE_DATA')
PRICE_DATA

'../../05_src/data/prices/'

In [5]:
# Find a path for all parquet files
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)
parquet_files

['../../05_src/data/prices\\ACN\\ACN_2002\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2002\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2003\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2003\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2004\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2004\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2005\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2005\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2006\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2006\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2007\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2007\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2008\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2008\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2009\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2009\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_201

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [ ]:
# Write your code below.

# Read the parquet files  as  dask dataframe
dd_px = dd.read_parquet(parquet_files).set_index("ticker")




In [41]:
# Add lag for variable Close

dd_lags_shift = dd_px.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(Close_lag_1 = x['Close'].shift(1), Adj_Close_lag_1 = x['Adj Close'].shift(1))
)

C:\Users\AlexandreT2\AppData\Local\Temp\ipykernel_11892\1593936975.py:3: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_lags_shift = dd_px.groupby('ticker', group_keys=False).apply(


In [42]:
#Calculate returns and hi-lo range

dd_feat = dd_lags_shift.assign(
    returns = lambda x: x['Close']/x['Close_lag_1'] - 1,
    hi_lo_range = lambda x: x['High'] -x['Low']
)

dd_feat

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
npartitions=60,,,,,,,,,,,,,
ACN,datetime64[ns],float64,float64,float64,float64,float64,float64,object,int32,float64,float64,float64,float64
ALDX,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...,...,...,...,...


In [43]:
dd_feat.compute()

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
ticker,,,,,,,,,,,,,
ACN,2002-01-02,26.900000,26.910000,25.950001,26.209999,19.703959,900500.0,ACN.csv,2002,NaN,NaN,NaN,0.959999
ACN,2002-01-03,26.230000,26.270000,25.299999,25.389999,19.087507,698200.0,ACN.csv,2002,26.209999,19.703959,-0.031286,0.970001
ACN,2002-01-04,25.389999,28.200001,25.240000,27.700001,20.824106,2708500.0,ACN.csv,2002,25.389999,19.087507,0.090981,2.960001
ACN,2002-01-07,27.770000,27.799999,26.450001,26.450001,19.884382,1113600.0,ACN.csv,2002,27.700001,20.824106,-0.045126,1.349998
ACN,2002-01-08,26.459999,27.299999,24.120001,27.280001,20.508362,1746900.0,ACN.csv,2002,26.450001,19.884382,0.031380,3.179998
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,2020-02-11,7.640000,7.850000,7.320000,7.350000,7.350000,491700.0,ZIXI.csv,2020,7.610000,7.610000,-0.034166,0.530000
ZIXI,2020-02-12,7.380000,7.400000,7.110000,7.290000,7.290000,358600.0,ZIXI.csv,2020,7.350000,7.350000,-0.008163,0.290000
ZIXI,2020-02-13,7.230000,7.310000,7.150000,7.160000,7.160000,314500.0,ZIXI.csv,2020,7.290000,7.290000,-0.017833,0.160000


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [57]:
# Write your code below.

# Converting dask dataframe "dd_feat" to pandas dataframe

dd_feat_pandas_df = dd_feat.compute()
print(dd_feat_pandas_df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 239548 entries, ACN to ZIXI
Data columns (total 13 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   Date             239548 non-null  datetime64[ns]
 1   Open             239545 non-null  float64       
 2   High             239545 non-null  float64       
 3   Low              239545 non-null  float64       
 4   Close            239545 non-null  float64       
 5   Adj Close        239545 non-null  float64       
 6   Volume           239545 non-null  float64       
 7   source           239548 non-null  object        
 8   Year             239548 non-null  int32         
 9   Close_lag_1      239485 non-null  float64       
 10  Adj_Close_lag_1  239485 non-null  float64       
 11  returns          239482 non-null  float64       
 12  hi_lo_range      239545 non-null  float64       
dtypes: datetime64[ns](1), float64(10), int32(1), object(1)
memory usage: 24.7+ MB
N

In [58]:
dd_feat_pandas_df.reset_index()

,ticker,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
0,ACN,2002-01-02,26.900000,26.910000,25.950001,26.209999,19.703959,900500.0,ACN.csv,2002,NaN,NaN,NaN,0.959999
1,ACN,2002-01-03,26.230000,26.270000,25.299999,25.389999,19.087507,698200.0,ACN.csv,2002,26.209999,19.703959,-0.031286,0.970001
2,ACN,2002-01-04,25.389999,28.200001,25.240000,27.700001,20.824106,2708500.0,ACN.csv,2002,25.389999,19.087507,0.090981,2.960001
3,ACN,2002-01-07,27.770000,27.799999,26.450001,26.450001,19.884382,1113600.0,ACN.csv,2002,27.700001,20.824106,-0.045126,1.349998
4,ACN,2002-01-08,26.459999,27.299999,24.120001,27.280001,20.508362,1746900.0,ACN.csv,2002,26.450001,19.884382,0.031380,3.179998
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239543,ZIXI,2020-02-11,7.640000,7.850000,7.320000,7.350000,7.350000,491700.0,ZIXI.csv,2020,7.610000,7.610000,-0.034166,0.530000
239544,ZIXI,2020-02-12,7.380000,7.400000,7.110000,7.290000,7.290000,358600.0,ZIXI.csv,2020,7.350000,7.350000,-0.008163,0.290000
239545,ZIXI,2020-02-13,7.230000,7.310000,7.150000,7.160000,7.160000,314500.0,ZIXI.csv,2020,7.290000,7.290000,-0.017833,0.160000
239546,ZIXI,2020-02-14,7.130000,7.260000,7.110000,7.200000,7.200000,282100.0,ZIXI.csv,2020,7.160000,7.160000,0.005587,0.150000


In [59]:
#Adding a feature containing a moving average of "returns" using a meathod .rolling().mean()

dd_feat_mov_aver = dd_feat.assign(returns_10days_moving_aver = lambda x: x['returns'].rolling(window = 10).mean())
dd_feat_mov_aver.compute().reset_index()

,ticker,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range,returns_10days_moving_aver
0,ACN,2002-01-02,26.900000,26.910000,25.950001,26.209999,19.703959,900500.0,ACN.csv,2002,NaN,NaN,NaN,0.959999,NaN
1,ACN,2002-01-03,26.230000,26.270000,25.299999,25.389999,19.087507,698200.0,ACN.csv,2002,26.209999,19.703959,-0.031286,0.970001,NaN
2,ACN,2002-01-04,25.389999,28.200001,25.240000,27.700001,20.824106,2708500.0,ACN.csv,2002,25.389999,19.087507,0.090981,2.960001,NaN
3,ACN,2002-01-07,27.770000,27.799999,26.450001,26.450001,19.884382,1113600.0,ACN.csv,2002,27.700001,20.824106,-0.045126,1.349998,NaN
4,ACN,2002-01-08,26.459999,27.299999,24.120001,27.280001,20.508362,1746900.0,ACN.csv,2002,26.450001,19.884382,0.031380,3.179998,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239543,ZIXI,2020-02-11,7.640000,7.850000,7.320000,7.350000,7.350000,491700.0,ZIXI.csv,2020,7.610000,7.610000,-0.034166,0.530000,0.007201
239544,ZIXI,2020-02-12,7.380000,7.400000,7.110000,7.290000,7.290000,358600.0,ZIXI.csv,2020,7.350000,7.350000,-0.008163,0.290000,0.006239
239545,ZIXI,2020-02-13,7.230000,7.310000,7.150000,7.160000,7.160000,314500.0,ZIXI.csv,2020,7.290000,7.290000,-0.017833,0.160000,0.003437
239546,ZIXI,2020-02-14,7.130000,7.260000,7.110000,7.200000,7.200000,282100.0,ZIXI.csv,2020,7.160000,7.160000,0.005587,0.150000,0.006733


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.